# Task 5D: Object Detection using Azure Computer Vision
## SIG788 - Engineering AI Solutions

**Objective:** Detect objects in images using Azure Computer Vision API and evaluate the results.

#### STEP 1: Install and Import Required Libraries

In [2]:
# Install required packages
!pip install requests Pillow pandas --quiet

print(" Packages installed successfully")

 Packages installed successfully


In [4]:
# Import the packages

# Standard library imports
import os                    # For file operations
import json                  # For JSON data handling
from pathlib import Path     # For file path operations

# Third-party imports
import requests              # For making HTTP requests to Azure API
from PIL import Image, ImageDraw, ImageFont  # For image manipulation
import pandas as pd          # For creating data tables

# Jupyter-specific import
from IPython.display import display  # For displaying images in notebook

# Verify all imports were successful
print("All libraries imported successfully")
print("\nImported packages:")
print("  • requests - HTTP requests to Azure")
print("  • Pillow - Image processing")
print("  • pandas - Data organization")
print("  • IPython - Notebook display functions")

All libraries imported successfully

Imported packages:
  • requests - HTTP requests to Azure
  • Pillow - Image processing
  • pandas - Data organization
  • IPython - Notebook display functions


## STEP 2: Configure Azure Credentials and Validate Input

**What we're doing:**
- Store Azure API credentials (endpoint and key)
- Specify the image file to process
- Validate that all required information is provided

**Important:**
- ⚠️ **UPDATE CELL 2.1** with your actual Azure credentials before running
- Never share your API key publicly
- The image file must be in the same directory as this notebook

### STEP 2.1: Define Configuration Variables

**⚠️ IMPORTANT:** Replace the placeholder values with your actual Azure credentials

In [17]:
# Validate that credentials have been loaded from environment
print("Validating configuration...\n")

if not AZURE_ENDPOINT:
    raise ValueError("AZURE_ENDPOINT not set. Add it to your .env file.")
else:
    print(f"✓ Azure endpoint configured: {AZURE_ENDPOINT}")

if not AZURE_API_KEY:
    raise ValueError("AZURE_API_KEY not set. Add it to your .env file.")
else:
    print(f"✓ API key configured ({len(AZURE_API_KEY)} characters)")

import os
if not os.path.exists(IMAGE_FILE_PATH):
    raise FileNotFoundError(f"Image file '{IMAGE_FILE_PATH}' not found. Place it in the same directory as this notebook.")
else:
    file_size_mb = os.path.getsize(IMAGE_FILE_PATH) / (1024 * 1024)
    print(f"✓ Image file found: {IMAGE_FILE_PATH} ({file_size_mb:.2f} MB)")

print("\n✅ All configuration validated successfully!")


Configuration variables defined
  Endpoint: https://cv-task5-sahid.cognitiveservices.azure.com/
  API Key: 5OYUOVdp1KpG8lvUU1YOmNqfL1LObTHWQNudlnXfuXcaFq4751mTJQQJ99CDACGhslBXJ3w3AAAFACOGV8Ak
  Image: test_image.jpg


### STEP 2.2: Validate Configuration

**What we're doing:**
- Check that Azure credentials were entered (not placeholders)
- Verify the image file exists in the directory
- Show validation results

In [19]:
# Validate that credentials have been loaded from environment
print("Validating configuration...\n")

if not AZURE_ENDPOINT:
    raise ValueError("AZURE_ENDPOINT not set. Add it to your .env file.")
else:
    print(f"✓ Azure endpoint configured: {AZURE_ENDPOINT}")

if not AZURE_API_KEY:
    raise ValueError("AZURE_API_KEY not set. Add it to your .env file.")
else:
    print(f"✓ API key configured ({len(AZURE_API_KEY)} characters)")

import os
if not os.path.exists(IMAGE_FILE_PATH):
    raise FileNotFoundError(f"Image file '{IMAGE_FILE_PATH}' not found. Place it in the same directory as this notebook.")
else:
    file_size_mb = os.path.getsize(IMAGE_FILE_PATH) / (1024 * 1024)
    print(f"✓ Image file found: {IMAGE_FILE_PATH} ({file_size_mb:.2f} MB)")

print("\n✅ All configuration validated successfully!")


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (3840018168.py, line 22)

## STEP 3: Load and Display the Test Image

**What we're doing:**
- Open the test image file
- Check its properties (size, format, etc.)
- Display it in the notebook so we can see what we're analyzing

In [ ]:
# Load the image file
print(f"Loading image: {IMAGE_FILE_PATH}\n")

try:
    # Open image using Pillow
    original_image = Image.open(IMAGE_FILE_PATH)
    
    # Get image properties
    image_width = original_image.width
    image_height = original_image.height
    image_format = original_image.format
    image_mode = original_image.mode
    
    print("✓ Image loaded successfully")
    print(f"\nImage Properties:")
    print(f"  Width:  {image_width} pixels")
    print(f"  Height: {image_height} pixels")
    print(f"  Format: {image_format}")
    print(f"  Color Mode: {image_mode}")
    
except Exception as e:
    print(f"❌ Error loading image: {str(e)}")
    raise

In [ ]:
# Display the original image in the notebook
# Resize for better display if very large

print("Displaying original image...\n")

# Calculate display size (max 800px width for readability)
display_width = min(800, image_width)
display_height = int(display_width * image_height / image_width)

# Create resized copy for display
display_image = original_image.resize((display_width, display_height), Image.Resampling.LANCZOS)

# Show the image
display(display_image)

print(f"\n(Displayed at {display_width}×{display_height}px, original is {image_width}×{image_height}px)")

## STEP 4: Call Azure Computer Vision API

**What we're doing:**
- Construct the API endpoint URL
- Prepare request headers with authentication
- Read the image file as binary data
- Send HTTP POST request to Azure API
- Receive and parse the JSON response

**Technical Details:**
- API version: v3.2 (detect endpoint)
- HTTP method: POST
- Request body: Raw image file (binary)
- Authentication: API key in header
- Response: JSON with detected objects and bounding boxes

In [ ]:
# Construct the API endpoint URL
# Format: https://[region].cognitiveservices.azure.com/vision/v3.2/detect

api_endpoint = f"{AZURE_ENDPOINT.rstrip('/')}/vision/v3.2/detect"

print(f"API Endpoint: {api_endpoint}\n")

# Prepare HTTP request headers
# These headers authenticate our request and specify the content type

request_headers = {
    # Subscription key for authentication
    "Ocp-Apim-Subscription-Key": AZURE_API_KEY,
    # Content type: sending raw image file
    "Content-Type": "application/octet-stream"
}

print("Request Headers:")
print(f"  Ocp-Apim-Subscription-Key: [hidden]")
print(f"  Content-Type: application/octet-stream")

In [ ]:
# Read the image file as binary data
print(f"\nReading image file as binary data...")

try:
    with open(IMAGE_FILE_PATH, "rb") as image_file:
        image_binary_data = image_file.read()
    
    print(f"✓ Image read successfully")
    print(f"  Size: {len(image_binary_data):,} bytes")
    
except Exception as e:
    print(f"❌ Error reading image: {str(e)}")
    raise

In [ ]:
# Send HTTP POST request to Azure Computer Vision API
print(f"\n🔍 Calling Azure Computer Vision API...")
print(f"   This may take 2-5 seconds...\n")

try:
    # Make the POST request
    api_response = requests.post(
        url=api_endpoint,
        headers=request_headers,
        data=image_binary_data,
        timeout=30  # 30 second timeout
    )
    
    print(f"✓ API request completed")
    print(f"  HTTP Status Code: {api_response.status_code}")
    
except requests.exceptions.Timeout:
    print(f"❌ ERROR: Request timeout (exceeded 30 seconds)")
    print(f"   The API took too long to respond")
    raise
except requests.exceptions.ConnectionError:
    print(f"❌ ERROR: Connection error")
    print(f"   Could not connect to Azure API")
    print(f"   Check your internet connection and endpoint URL")
    raise
except Exception as e:
    print(f"❌ ERROR: {str(e)}")
    raise

In [ ]:
# Check if the API request was successful
print(f"\nChecking API response status...\n")

if api_response.status_code != 200:
    print(f"❌ ERROR: API returned status code {api_response.status_code}")
    print(f"\nResponse:")
    print(api_response.text)
    print(f"\nTroubleshooting:")
    
    if api_response.status_code == 401:
        print(f"  • 401 Unauthorized: Your API key is invalid or expired")
        print(f"    - Get a new key from Azure Portal")
    elif api_response.status_code == 403:
        print(f"  • 403 Forbidden: Your resource may be in 'Stopped' state")
        print(f"    - Check Azure Portal that resource is 'Running'")
    else:
        print(f"  • Check your endpoint URL and API key")
    
    raise Exception(f"API request failed with status code {api_response.status_code}")

else:
    print(f"✅ API request successful!")
    print(f"   Status: 200 OK")

In [ ]:
# Parse the JSON response from API
print(f"\nParsing API response...\n")

try:
    # Convert response text to Python dictionary
    api_response_data = api_response.json()
    
    # Extract the list of detected objects
    detected_objects_list = api_response_data.get("objects", [])
    
    # Extract image metadata
    image_metadata = api_response_data.get("metadata", {})
    
    print(f"✓ Response parsed successfully")
    print(f"\nDetection Summary:")
    print(f"  Objects detected: {len(detected_objects_list)}")
    print(f"  Image size: {image_metadata.get('width', '?')} × {image_metadata.get('height', '?')} pixels")
    
except Exception as e:
    print(f"❌ Error parsing response: {str(e)}")
    print(f"\nRaw response:")
    print(api_response.text)
    raise

## STEP 5: Parse and Analyze Detection Results

**What we're doing:**
- Extract individual detection results from API response
- Organize data into a structured format
- Display detection results in a clear table
- Calculate statistics (confidence, etc.)

In [ ]:
# Extract and organize detection results
print(f"Processing {len(detected_objects_list)} detected objects...\n")

# Create a list to store processed detection data
processed_detections = []

# Process each detected object
for object_index, detected_object in enumerate(detected_objects_list, 1):
    # Extract object properties from API response
    object_name = detected_object["object"]
    confidence_score = detected_object["confidence"]
    bounding_box = detected_object["rectangle"]
    
    # Extract bounding box coordinates
    bbox_x = bounding_box["x"]
    bbox_y = bounding_box["y"]
    bbox_width = bounding_box["w"]
    bbox_height = bounding_box["h"]
    
    # Calculate the bottom-right corner of bounding box
    bbox_x2 = bbox_x + bbox_width
    bbox_y2 = bbox_y + bbox_height
    
    # Store processed data
    detection_entry = {
        "Detection_ID": object_index,
        "Object_Name": object_name,
        "Confidence_Score": confidence_score,
        "Confidence_Percentage": round(confidence_score * 100, 2),
        "Bounding_Box_X1": bbox_x,
        "Bounding_Box_Y1": bbox_y,
        "Bounding_Box_X2": bbox_x2,
        "Bounding_Box_Y2": bbox_y2,
        "Box_Width": bbox_width,
        "Box_Height": bbox_height
    }
    
    processed_detections.append(detection_entry)

print(f"✓ Processed {len(processed_detections)} objects")

In [ ]:
# Check if any objects were detected
if len(processed_detections) == 0:
    print("\n⚠️  WARNING: No objects detected in the image!")
    print("\nPossible reasons:")
    print("  • Image is too blurry")
    print("  • Image has poor lighting")
    print("  • Objects in image are not standard types")
    print("  • Image quality is too low")
    print("\nSuggestions for next attempt:")
    print("  • Use an image with 2-5+ distinct, clear objects")
    print("  • Try a street scene, room, or nature photo")
    print("  • Ensure good lighting and clear visibility")
else:
    print("\n✓ Objects successfully detected!")
    print(f"\nDetailed Detection Results:\n")
    
    # Print header
    print("="*100)
    print(f"{'#':<3} {'Object':<20} {'Confidence':<15} {'Bounding Box':<50}")
    print("="*100)
    
    # Print each detection
    for detection in processed_detections:
        detection_id = detection["Detection_ID"]
        object_name = detection["Object_Name"]
        confidence_pct = detection["Confidence_Percentage"]
        x1 = detection["Bounding_Box_X1"]
        y1 = detection["Bounding_Box_Y1"]
        x2 = detection["Bounding_Box_X2"]
        y2 = detection["Bounding_Box_Y2"]
        
        bbox_str = f"({x1}, {y1}) to ({x2}, {y2})"
        print(f"{detection_id:<3} {object_name:<20} {confidence_pct:.1f}%{'':<10} {bbox_str:<50}")
    
    print("="*100)

In [ ]:
# Calculate detection statistics
if len(processed_detections) > 0:
    print("\n📊 Detection Statistics:\n")
    
    # Extract confidence scores
    confidence_scores = [d["Confidence_Percentage"] for d in processed_detections]
    
    # Calculate statistics
    average_confidence = sum(confidence_scores) / len(confidence_scores)
    highest_confidence = max(confidence_scores)
    lowest_confidence = min(confidence_scores)
    
    print(f"  Total Objects Detected:    {len(processed_detections)}")
    print(f"  Average Confidence:        {average_confidence:.2f}%")
    print(f"  Highest Confidence:        {highest_confidence:.2f}%")
    print(f"  Lowest Confidence:         {lowest_confidence:.2f}%")
    
    # Count objects by type
    object_type_counts = {}
    for detection in processed_detections:
        obj_type = detection["Object_Name"]
        object_type_counts[obj_type] = object_type_counts.get(obj_type, 0) + 1
    
    print(f"\n  Objects by Type:")
    for obj_type, count in sorted(object_type_counts.items()):
        print(f"    • {obj_type}: {count}")

## STEP 6: Visualize Detections with Bounding Boxes

**What we're doing:**
- Create a copy of the original image
- Draw colored bounding boxes around each detected object
- Add labels showing object name and confidence percentage
- Save the annotated image
- Display it in the notebook

In [ ]:
if len(processed_detections) > 0:
    print("\n🎨 Drawing bounding boxes...\n")
    
    # Load the original image again for annotation
    annotated_image = Image.open(IMAGE_FILE_PATH).convert("RGB")
    
    # Create drawing object
    draw_context = ImageDraw.Draw(annotated_image)
    
    # Try to load a nice font, fallback to default if not available
    try:
        label_font = ImageFont.truetype("arial.ttf", 20)
    except:
        try:
            label_font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 20)
        except:
            label_font = ImageFont.load_default()
    
    print(f"✓ Drawing {len(processed_detections)} bounding boxes...")
    
    # Draw a bounding box for each detection
    for detection_index, detection in enumerate(processed_detections):
        # Get color for this detection (cycle through color palette)
        color_hex = BOUNDING_BOX_COLORS[detection_index % len(BOUNDING_BOX_COLORS)]
        
        # Get detection information
        object_name = detection["Object_Name"]
        confidence_pct = detection["Confidence_Percentage"]
        
        # Get bounding box coordinates
        x1 = detection["Bounding_Box_X1"]
        y1 = detection["Bounding_Box_Y1"]
        x2 = detection["Bounding_Box_X2"]
        y2 = detection["Bounding_Box_Y2"]
        
        # Draw rectangle (3 pixels thick for visibility)
        line_width = 3
        for line_offset in range(line_width):
            draw_context.rectangle(
                [(x1 - line_offset, y1 - line_offset), (x2 + line_offset, y2 + line_offset)],
                outline=color_hex
            )
        
        # Create label text
        label_text = f"{object_name} {confidence_pct:.1f}%"
        
        # Calculate label position (above bounding box)
        label_x = x1
        label_y = y1 - 30 if y1 > 30 else y1 + 5
        
        # Get text dimensions
        text_bbox = draw_context.textbbox((label_x, label_y), label_text, font=label_font)
        
        # Draw label background rectangle
        label_bg_padding = 5
        draw_context.rectangle(
            [
                (text_bbox[0] - label_bg_padding, text_bbox[1] - label_bg_padding),
                (text_bbox[2] + label_bg_padding, text_bbox[3] + label_bg_padding)
            ],
            fill=color_hex
        )
        
        # Draw label text
        draw_context.text((label_x, label_y), label_text, fill="white", font=label_font)
    
    print(f"✓ Bounding boxes drawn successfully")

else:
    print("Skipped (no objects detected)")

In [ ]:
if len(processed_detections) > 0:
    # Save the annotated image
    output_image_path = "detected_objects_output.jpg"
    annotated_image.save(output_image_path, quality=95)
    
    print(f"✓ Annotated image saved: {output_image_path}\n")
    
    # Display the annotated image
    print(f"📸 Displaying annotated image:\n")
    
    # Resize for display if needed
    display_width = min(900, annotated_image.width)
    display_height = int(display_width * annotated_image.height / annotated_image.width)
    display_image_annotated = annotated_image.resize((display_width, display_height), Image.Resampling.LANCZOS)
    
    display(display_image_annotated)
else:
    print("Skipped (no objects detected)")

## STEP 7: Save Results in Multiple Formats

**What we're doing:**
- Create a pandas DataFrame from detection results
- Export to CSV format (for Excel compatibility)
- Export to JSON format (for structured data)

**Why multiple formats:**
- CSV: Easy to open in Excel, use in reports
- JSON: Structured data, easy to parse programmatically
- Both formats preserve all detection information

In [ ]:
if len(processed_detections) > 0:
    print("\n💾 Saving results in multiple formats...\n")
    
    # Create pandas DataFrame from detection results
    # Select only the most important columns for the table
    results_dataframe = pd.DataFrame([
        {
            "ID": d["Detection_ID"],
            "Object_Name": d["Object_Name"],
            "Confidence_%": d["Confidence_Percentage"],
            "X1": d["Bounding_Box_X1"],
            "Y1": d["Bounding_Box_Y1"],
            "X2": d["Bounding_Box_X2"],
            "Y2": d["Bounding_Box_Y2"],
            "Width": d["Box_Width"],
            "Height": d["Box_Height"]
        }
        for d in processed_detections
    ])
    
    # Save to CSV
    csv_output_path = "detection_results.csv"
    results_dataframe.to_csv(csv_output_path, index=False)
    print(f"✓ CSV file saved: {csv_output_path}")
    
    # Display the CSV content in the notebook
    print(f"\nCSV Content:")
    print(results_dataframe.to_string())
else:
    print("Skipped (no objects detected)")

In [ ]:
if len(processed_detections) > 0:
    # Save to JSON format
    
    json_results = {
        "analysis_metadata": {
            "image_file": IMAGE_FILE_PATH,
            "image_dimensions": {
                "width_pixels": image_width,
                "height_pixels": image_height
            },
            "api_endpoint": api_endpoint,
            "api_version": "v3.2"
        },
        "detection_summary": {
            "total_objects_detected": len(processed_detections),
            "average_confidence": round(sum([d["Confidence_Percentage"] for d in processed_detections]) / len(processed_detections), 2),
            "highest_confidence": round(max([d["Confidence_Percentage"] for d in processed_detections]), 2),
            "lowest_confidence": round(min([d["Confidence_Percentage"] for d in processed_detections]), 2)
        },
        "detections": processed_detections
    }
    
    json_output_path = "detection_results.json"
    with open(json_output_path, "w") as json_file:
        json.dump(json_results, json_file, indent=2)
    
    print(f"✓ JSON file saved: {json_output_path}")
    print(f"\nJSON Content (formatted):")
    print(json.dumps(json_results, indent=2))
else:
    print("Skipped (no objects detected)")

## STEP 8: Generate Summary Report

**What we're doing:**
- Create a comprehensive summary of the entire analysis
- List all input parameters and results
- Show which output files were created
- Provide next steps for the assignment

In [ ]:
# Generate final summary report

print("\n" + "="*100)
print("OBJECT DETECTION ANALYSIS SUMMARY".center(100))
print("Task 5D - SIG788 Engineering AI Solutions".center(100))
print("="*100)

print(f"\n📋 INPUT INFORMATION:")
print(f"  Image File:          {IMAGE_FILE_PATH}")
print(f"  Image Dimensions:    {image_width} × {image_height} pixels")
print(f"  API Endpoint:        {AZURE_ENDPOINT}")
print(f"  API Version:         v3.2 (detect)")

if len(processed_detections) > 0:
    print(f"\n✅ DETECTION RESULTS:")
    print(f"  Objects Detected:    {len(processed_detections)}")
    print(f"  Average Confidence:  {sum([d['Confidence_Percentage'] for d in processed_detections]) / len(processed_detections):.2f}%")
    print(f"  Highest Confidence:  {max([d['Confidence_Percentage'] for d in processed_detections]):.2f}%")
    print(f"  Lowest Confidence:   {min([d['Confidence_Percentage'] for d in processed_detections]):.2f}%")
    
    print(f"\n📁 OUTPUT FILES CREATED:")
    print(f"  1. detected_objects_output.jpg")
    print(f"     └─ Image with drawn bounding boxes")
    print(f"     └─ Use in: Task 5D report, Results section")
    
    print(f"\n  2. detection_results.csv")
    print(f"     └─ Table format (can open in Excel)")
    print(f"     └─ Use in: Task 5D report, Appendix or Results")
    
    print(f"\n  3. detection_results.json")
    print(f"     └─ Structured data format")
    print(f"     └─ Use in: Analysis and statistics")
    
    print(f"\n🎯 DETECTED OBJECTS:")
    for detection in processed_detections:
        print(f"  • {detection['Object_Name']:<20} Confidence: {detection['Confidence_Percentage']:>6.2f}%")
    
    print(f"\n📝 NEXT STEPS FOR TASK 5D:")
    print(f"  1. Download the output files (JPG, CSV, JSON)")
    print(f"  2. Insert annotated image into your report (Results section)")
    print(f"  3. Include detection table in your report")
    print(f"  4. Analyze accuracy and confidence in Evaluation section")
    print(f"  5. Discuss strengths and limitations of Azure Computer Vision")
    print(f"  6. Suggest improvements (fine-tuning, hybrid approaches, etc.)")
    
    print(f"\n✨ Ready for Task 5D submission!")

else:
    print(f"\n⚠️  NO OBJECTS DETECTED")
    print(f"\n  Please try again with:")
    print(f"    • A different image")
    print(f"    • An image with 2-5+ distinct, clear objects")
    print(f"    • Good lighting and visibility")

print("\n" + "="*100)
print("✅ Analysis Complete!".center(100))
print("="*100)